# Step 2. 강원도 산불발생 날씨·지수 대조군 재분석

산불 발생과 비발생 비교를 위한 사건, 격자, 누수 방지 기상, 캐나다 지수를 준비한다.

## 노트북 구성

- `0. 공통 설정과 데이터 로딩`: S2-01부터 S2-10까지 이어서 재사용하는 입력 경로, 원천 파일, 정제 산불, 격자, 시간기상 키를 준비한다.
- `S2-01`: 정제 산불·분석 단위·공간조인 감사를 수행한다.
- `S2-02` 이후 분석은 `S2-01 종료 / S2-02 추가 위치` 아래에 이어서 추가한다.

결과 해석은 이 노트북에 작성하지 않고 대응 `진행예정로그.md`에만 기록한다.

## 0. 공통 설정과 입력 경로 (S2 전체 공통)

In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

ANALYSIS_ID = "S2-01"
BOUNDARY_DISTANCE_M = 100.0

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    path
    for path in [cwd, *cwd.parents]
    if (path / "data").exists() and (path / "jsw" / "강원_재_EDA").exists()
)
PROJECT_DIR = REPO_ROOT / "jsw" / "강원_재_EDA"
TABLE_DIR = PROJECT_DIR / "outputs" / "Step2" / "tables"
PLOT_DIR = PROJECT_DIR / "outputs" / "Step2" / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

FIRE_RAW_PATH = REPO_ROOT / "data" / "강원도_데이터" / "강원도_산불발생.csv"
FIRE_CLEAN_PATH = REPO_ROOT / "data" / "학습데이터" / "산불발생_정제.csv"
FIRE_NEAR_DUP_PATH = REPO_ROOT / "data" / "학습데이터" / "산불발생_준중복제외.csv"
GRID_PATH = REPO_ROOT / "data" / "강원도_날씨데이터" / "강원도날씨_격자.geojson"
CLIMATE_TYPE_PATH = REPO_ROOT / "data" / "강원도_날씨데이터" / "강원도날씨_기후지형유형_셀분류.csv"
WEATHER_DERIVED_PATH = REPO_ROOT / "data" / "학습데이터" / "기상_시간단위_파생.csv"

SOURCE_PATHS = {
    "raw_fire": FIRE_RAW_PATH,
    "clean_fire": FIRE_CLEAN_PATH,
    "near_duplicate_audit": FIRE_NEAR_DUP_PATH,
    "weather_grid": GRID_PATH,
    "climate_type": CLIMATE_TYPE_PATH,
    "hourly_weather_derived": WEATHER_DERIVED_PATH,
}

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


## 0-1. 원천 파일 존재 여부와 공통 데이터 로딩 (S2 전체 공통)

In [ ]:
source_audit = pd.DataFrame(
    [
        {
            "source": name,
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else np.nan,
            "path": str(path),
        }
        for name, path in SOURCE_PATHS.items()
    ]
)
display(source_audit)
assert source_audit["exists"].all(), source_audit.loc[~source_audit["exists"], "path"].tolist()

raw_fire = pd.read_csv(FIRE_RAW_PATH, encoding="utf-8-sig")
clean_fire = pd.read_csv(FIRE_CLEAN_PATH, encoding="utf-8-sig", parse_dates=["기준시각"])
near_duplicate_audit = pd.read_csv(FIRE_NEAR_DUP_PATH, encoding="utf-8-sig", parse_dates=["기준시각", "대표_기준시각"])
grid = gpd.read_file(GRID_PATH).to_crs("EPSG:4326")
climate_type = pd.read_csv(CLIMATE_TYPE_PATH, encoding="utf-8-sig")
weather_keys = pd.read_csv(
    WEATHER_DERIVED_PATH,
    encoding="utf-8-sig",
    usecols=["기상셀ID", "일시"],
    parse_dates=["일시"],
)

print(f"raw_fire rows: {len(raw_fire):,}")
print(f"clean_fire rows: {len(clean_fire):,}")
print(f"near_duplicate_audit rows: {len(near_duplicate_audit):,}")
print(f"weather key rows: {len(weather_keys):,}")
display(clean_fire.head())


## S2-01. 정제 산불·분석 단위·공간조인 감사

### S2-01-1. 정제 단계별 행 수 감사

In [ ]:
raw_fire = raw_fire.copy()
raw_fire["발생일시"] = pd.to_datetime(
    dict(year=raw_fire["연도"], month=raw_fire["월"], day=raw_fire["일"]),
    errors="coerce",
) + pd.to_timedelta(raw_fire["시간"].fillna(0).astype(int), unit="h")
raw_fire["기준시각"] = raw_fire["발생일시"].dt.floor("h")

raw_fire_gdf = gpd.GeoDataFrame(
    raw_fire,
    geometry=gpd.points_from_xy(raw_fire["경도"], raw_fire["위도"]),
    crs="EPSG:4326",
)
raw_joined = gpd.sjoin(
    raw_fire_gdf,
    grid[["기상셀ID", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")
raw_joined = pd.DataFrame(raw_joined.drop(columns=["geometry"], errors="ignore"))

spatial_matched_raw = raw_joined[raw_joined["기상셀ID"].notna()].copy()
raw_spatial_unmatched = raw_joined[raw_joined["기상셀ID"].isna()].copy()
after_exact = spatial_matched_raw.drop_duplicates(
    subset=["위도", "경도", "기준시각"],
    keep="first",
).reset_index(drop=True)

expected_clean_rows = len(after_exact) - len(near_duplicate_audit)
unique_exposures = clean_fire[["기상셀ID", "기준시각"]].drop_duplicates().copy()
unique_coord_time = clean_fire[["위도", "경도", "기준시각"]].drop_duplicates()

cleaning_stage_audit = pd.DataFrame(
    [
        {
            "stage": "원본 산불",
            "rows": len(raw_fire),
            "removed_from_previous": 0,
            "basis": "data/강원도_데이터/강원도_산불발생.csv",
        },
        {
            "stage": "기상셀 공간조인 성공",
            "rows": len(spatial_matched_raw),
            "removed_from_previous": len(raw_fire) - len(spatial_matched_raw),
            "basis": "원본 좌표가 92개 기상셀 polygon 안에 포함된 행",
        },
        {
            "stage": "완전중복 좌표-시각 제거 후",
            "rows": len(after_exact),
            "removed_from_previous": len(spatial_matched_raw) - len(after_exact),
            "basis": "subset=[위도, 경도, 기준시각], keep=first",
        },
        {
            "stage": "동시각 100m 준중복 제거 후 정제 사건",
            "rows": len(clean_fire),
            "removed_from_previous": len(near_duplicate_audit),
            "basis": "data/학습데이터/산불발생_준중복제외.csv 감사 행 수",
        },
        {
            "stage": "고유 기상 노출",
            "rows": len(unique_exposures),
            "removed_from_previous": len(clean_fire) - len(unique_exposures),
            "basis": "고유 기상셀ID × 기준시각",
        },
    ]
)

display(cleaning_stage_audit)


### S2-01-2. 분석 단위·키 품질·결측 감사

In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371000 * 2 * np.arcsin(np.sqrt(a))


def residual_near_duplicate_pairs(df, threshold_m):
    rows = []
    for 기준시각, group in df.groupby("기준시각", sort=False):
        if len(group) < 2:
            continue
        group = group.reset_index(drop=True)
        lon = group["경도"].to_numpy(dtype=float)
        lat = group["위도"].to_numpy(dtype=float)
        for i in range(len(group) - 1):
            dists = haversine_m(lon[i], lat[i], lon[i + 1 :], lat[i + 1 :])
            for offset in np.where(dists <= threshold_m)[0]:
                j = i + 1 + int(offset)
                rows.append(
                    {
                        "기준시각": 기준시각,
                        "fire_id_1": group.loc[i, "fire_id"],
                        "fire_id_2": group.loc[j, "fire_id"],
                        "distance_m": float(dists[offset]),
                    }
                )
    return pd.DataFrame(rows)


required_clean_columns = [
    "fire_id",
    "기준시각",
    "위도",
    "경도",
    "기상셀ID",
    "기후지형유형",
    "월_key",
    "시간_key",
    "샘플유형",
]
distance_columns = [col for col in clean_fire.columns if col.endswith("_최단거리_m")]

valid_cell_ids = set(grid["기상셀ID"])
climate_lookup = climate_type.set_index("기상셀ID")["기후지형유형"]
clean_with_climate = clean_fire.merge(
    climate_type.rename(columns={"기후지형유형": "기후지형유형_기준표"}),
    on="기상셀ID",
    how="left",
)

exposure_group_sizes = clean_fire.groupby(["기상셀ID", "기준시각"], dropna=False).size().rename("events_per_exposure").reset_index()
exposure_duplicate_distribution = (
    exposure_group_sizes["events_per_exposure"]
    .value_counts()
    .sort_index()
    .rename_axis("events_per_cell_time")
    .reset_index(name="exposure_count")
)
exposure_duplicate_distribution["event_rows"] = (
    exposure_duplicate_distribution["events_per_cell_time"] * exposure_duplicate_distribution["exposure_count"]
)
exposure_duplicate_distribution["share_of_exposures"] = (
    exposure_duplicate_distribution["exposure_count"] / exposure_duplicate_distribution["exposure_count"].sum()
)

weather_key_duplicates = int(weather_keys.duplicated(["기상셀ID", "일시"]).sum())
weather_key_set = weather_keys.rename(columns={"일시": "기준시각"})
exposure_weather_join = unique_exposures.merge(
    weather_key_set,
    on=["기상셀ID", "기준시각"],
    how="left",
    indicator=True,
)
missing_weather_exposures = exposure_weather_join[exposure_weather_join["_merge"] != "both"].copy()

residual_pairs = residual_near_duplicate_pairs(clean_fire, BOUNDARY_DISTANCE_M)

clean_fire_gdf = gpd.GeoDataFrame(
    clean_fire.copy(),
    geometry=gpd.points_from_xy(clean_fire["경도"], clean_fire["위도"]),
    crs="EPSG:4326",
)
grid_5186 = grid.to_crs("EPSG:5186")
clean_fire_5186 = clean_fire_gdf.to_crs("EPSG:5186")
cell_geometry = grid_5186.set_index("기상셀ID").geometry
clean_fire_5186["cell_boundary_distance_m"] = [
    point.distance(cell_geometry.loc[cell_id].boundary) if cell_id in cell_geometry.index else np.nan
    for point, cell_id in zip(clean_fire_5186.geometry, clean_fire_5186["기상셀ID"])
]
boundary_points = clean_fire_5186[clean_fire_5186["cell_boundary_distance_m"] <= BOUNDARY_DISTANCE_M].copy()
boundary_points_wgs84 = boundary_points.to_crs("EPSG:4326")

period_start = pd.Timestamp("2020-01-01 00:00:00")
period_end_exclusive = pd.Timestamp("2022-01-01 00:00:00")

clean_missing = clean_fire[required_clean_columns + distance_columns].isna().sum().reset_index()
clean_missing.columns = ["column", "missing_n"]
clean_missing["dataset"] = "clean_fire"
clean_missing["rows"] = len(clean_fire)
clean_missing["missing_pct"] = clean_missing["missing_n"] / len(clean_fire) * 100

raw_required_columns = ["fire_id", "연도", "월", "일", "시간", "위도", "경도", "기준시각"]
raw_missing = raw_fire[raw_required_columns].isna().sum().reset_index()
raw_missing.columns = ["column", "missing_n"]
raw_missing["dataset"] = "raw_fire"
raw_missing["rows"] = len(raw_fire)
raw_missing["missing_pct"] = raw_missing["missing_n"] / len(raw_fire) * 100

missingness_audit = pd.concat([raw_missing, clean_missing], ignore_index=True)[
    ["dataset", "column", "rows", "missing_n", "missing_pct"]
]

clean_fire_id_duplicate_rows = int(clean_fire.duplicated(["fire_id"], keep=False).sum())
clean_exact_coord_time_duplicate_rows = int(clean_fire.duplicated(["위도", "경도", "기준시각"], keep=False).sum())
clean_required_missing_total = int(clean_fire[required_clean_columns].isna().sum().sum())
invalid_cell_rows = int((~clean_fire["기상셀ID"].isin(valid_cell_ids)).sum())
climate_missing_rows = int(clean_with_climate["기후지형유형_기준표"].isna().sum())
climate_mismatch_rows = int(
    (
        clean_with_climate["기후지형유형_기준표"].notna()
        & (clean_with_climate["기후지형유형"] != clean_with_climate["기후지형유형_기준표"])
    ).sum()
)
out_of_period_rows = int(((clean_fire["기준시각"] < period_start) | (clean_fire["기준시각"] >= period_end_exclusive)).sum())

ys0071 = clean_fire[clean_fire["기상셀ID"] == "YS_0071"].copy()
ys0071_audit = pd.DataFrame(
    [
        {
            "기상셀ID": "YS_0071",
            "event_rows": len(ys0071),
            "unique_weather_exposures": ys0071[["기상셀ID", "기준시각"]].drop_duplicates().shape[0],
            "unique_dates": ys0071["기준시각"].dt.date.nunique(),
            "min_time": ys0071["기준시각"].min(),
            "max_time": ys0071["기준시각"].max(),
        }
    ]
)

population_validation = pd.DataFrame(
    [
        {"metric": "analysis_id", "value": ANALYSIS_ID, "notes": "정제 산불·분석 단위·공간조인 감사"},
        {"metric": "period", "value": "2020-01-01 <= 기준시각 < 2022-01-01", "notes": "Step1 고정 범위"},
        {"metric": "weather_cells_total", "value": int(grid["기상셀ID"].nunique()), "notes": "Step1 모집단 셀 수"},
        {"metric": "raw_fire_rows", "value": len(raw_fire), "notes": "원본 산불 행"},
        {"metric": "clean_fire_event_rows", "value": len(clean_fire), "notes": "정제 사건 단위 행"},
        {"metric": "control_rows", "value": 0, "notes": "S2-01에서는 대조군을 아직 생성하지 않음"},
        {"metric": "unique_fire_id", "value": int(clean_fire["fire_id"].nunique()), "notes": "정제 사건 고유 ID"},
        {"metric": "unique_coordinate_time", "value": len(unique_coord_time), "notes": "고유 위도·경도·기준시각"},
        {"metric": "unique_cell_time_exposures", "value": len(unique_exposures), "notes": "고유 기상셀ID×기준시각"},
        {"metric": "duplicate_cell_time_extra_event_rows", "value": len(clean_fire) - len(unique_exposures), "notes": "기상 비교에서는 중복 표본으로 쓰지 않을 행 수"},
        {"metric": "required_key_missing_total", "value": clean_required_missing_total, "notes": "정제 사건 필수 키 결측 총합"},
        {"metric": "clean_fire_id_duplicate_rows", "value": clean_fire_id_duplicate_rows, "notes": "fire_id 중복 행"},
        {"metric": "raw_spatial_unmatched_rows", "value": len(raw_spatial_unmatched), "notes": "원본 중 기상셀 polygon within 실패"},
        {"metric": "boundary_points_le_100m", "value": len(boundary_points), "notes": "정제 사건 중 matched cell 경계 100m 이내"},
        {"metric": "YS_0071_event_rows", "value": len(ys0071), "notes": "Step1 품질검토대상 셀 사건 수"},
        {"metric": "YS_0071_unique_exposures", "value": ys0071_audit.loc[0, "unique_weather_exposures"], "notes": "Step1 품질검토대상 셀 고유 노출 수"},
    ]
)

key_quality_audit = pd.DataFrame(
    [
        {"check_name": "pipeline_stage_row_count_match", "value": int(len(clean_fire) == expected_clean_rows), "pass": bool(len(clean_fire) == expected_clean_rows), "details": f"expected={expected_clean_rows}, actual={len(clean_fire)}"},
        {"check_name": "clean_fire_id_duplicate_rows", "value": clean_fire_id_duplicate_rows, "pass": clean_fire_id_duplicate_rows == 0, "details": "정제 fire_id 중복"},
        {"check_name": "clean_required_key_missing_total", "value": clean_required_missing_total, "pass": clean_required_missing_total == 0, "details": ",".join(required_clean_columns)},
        {"check_name": "clean_exact_coordinate_time_duplicate_rows", "value": clean_exact_coord_time_duplicate_rows, "pass": clean_exact_coord_time_duplicate_rows == 0, "details": "위도·경도·기준시각 완전중복"},
        {"check_name": "residual_near_duplicate_pairs_100m", "value": len(residual_pairs), "pass": len(residual_pairs) == 0, "details": "정제 후 동일시각 100m 이내 잔존 쌍"},
        {"check_name": "invalid_weather_cell_rows", "value": invalid_cell_rows, "pass": invalid_cell_rows == 0, "details": "92개 기상셀ID 범위 밖"},
        {"check_name": "climate_type_missing_rows", "value": climate_missing_rows, "pass": climate_missing_rows == 0, "details": "기후지형유형 기준표 미결합"},
        {"check_name": "climate_type_mismatch_rows", "value": climate_mismatch_rows, "pass": climate_mismatch_rows == 0, "details": "정제 파일과 기준표 기후지형유형 불일치"},
        {"check_name": "out_of_2020_2021_period_rows", "value": out_of_period_rows, "pass": out_of_period_rows == 0, "details": "Step1 고정 기간 위반"},
        {"check_name": "hourly_weather_key_duplicate_rows", "value": weather_key_duplicates, "pass": weather_key_duplicates == 0, "details": "기상_시간단위_파생 기상셀ID×일시 중복"},
        {"check_name": "unique_exposures_missing_hourly_weather", "value": len(missing_weather_exposures), "pass": len(missing_weather_exposures) == 0, "details": "고유 기상 노출의 시간기상 결합 실패"},
        {"check_name": "duplicate_cell_time_extra_event_rows_recorded", "value": len(clean_fire) - len(unique_exposures), "pass": True, "details": "사건 단위에는 유지, 기상 비교에서는 unique exposure 사용"},
    ]
)

sample_group_counts = (
    clean_fire["샘플유형"]
    .value_counts(dropna=False)
    .rename_axis("sample_group")
    .reset_index(name="rows")
)
sample_group_counts = pd.concat(
    [
        sample_group_counts,
        pd.DataFrame([{"sample_group": "Target_0_control_not_created_in_S2_01", "rows": 0}]),
    ],
    ignore_index=True,
)

raw_unmatched_records = raw_spatial_unmatched.copy()
raw_unmatched_records["audit_type"] = "raw_spatial_unmatched"
raw_unmatched_records["reason"] = "원본 좌표가 기상셀 polygon within 조건에 매칭되지 않음"
raw_unmatched_records["cell_boundary_distance_m"] = np.nan

final_out_of_range = clean_fire[
    (~clean_fire["기상셀ID"].isin(valid_cell_ids))
    | (clean_fire["기준시각"] < period_start)
    | (clean_fire["기준시각"] >= period_end_exclusive)
].copy()
final_out_of_range["audit_type"] = "clean_out_of_range"
final_out_of_range["reason"] = "정제 파일의 셀 또는 기간 범위 위반"
final_out_of_range["cell_boundary_distance_m"] = np.nan

boundary_records = pd.DataFrame(boundary_points_wgs84.drop(columns=["geometry"], errors="ignore")).copy()
boundary_records["audit_type"] = f"clean_cell_boundary_le_{int(BOUNDARY_DISTANCE_M)}m"
boundary_records["reason"] = "matched 기상셀 경계 100m 이내"

audit_record_columns = [
    "audit_type",
    "reason",
    "fire_id",
    "기준시각",
    "위도",
    "경도",
    "기상셀ID",
    "기후지형유형",
    "cell_boundary_distance_m",
]
unmatched_boundary_out_of_range = pd.concat(
    [raw_unmatched_records, final_out_of_range, boundary_records],
    ignore_index=True,
    sort=False,
)
for col in audit_record_columns:
    if col not in unmatched_boundary_out_of_range.columns:
        unmatched_boundary_out_of_range[col] = np.nan
unmatched_boundary_out_of_range = unmatched_boundary_out_of_range[audit_record_columns].sort_values(
    ["audit_type", "기준시각", "fire_id"],
    na_position="last",
)

unique_fire_exposures = exposure_group_sizes.merge(
    clean_fire.groupby(["기상셀ID", "기준시각"], dropna=False)
    .agg(
        fire_ids=("fire_id", lambda values: "|".join(map(str, values))),
        first_latitude=("위도", "first"),
        first_longitude=("경도", "first"),
        climate_type=("기후지형유형", "first"),
        month=("월_key", "first"),
        hour=("시간_key", "first"),
    )
    .reset_index(),
    on=["기상셀ID", "기준시각"],
    how="left",
)

tables = {
    "S2-01_source_audit.csv": source_audit,
    "S2-01_population_validation.csv": population_validation,
    "S2-01_cleaning_stage_audit.csv": cleaning_stage_audit,
    "S2-01_key_quality_audit.csv": key_quality_audit,
    "S2-01_missingness_audit.csv": missingness_audit,
    "S2-01_sample_group_counts.csv": sample_group_counts,
    "S2-01_exposure_duplicate_distribution.csv": exposure_duplicate_distribution,
    "S2-01_unmatched_boundary_out_of_range.csv": unmatched_boundary_out_of_range,
    "S2-01_ys0071_audit.csv": ys0071_audit,
    "S2-01_residual_near_duplicate_pairs.csv": residual_pairs,
    "S2-01_unique_weather_exposures.csv": unique_fire_exposures,
}

for filename, df in tables.items():
    df.to_csv(TABLE_DIR / filename, index=False, encoding="utf-8-sig")

display(population_validation)
display(key_quality_audit)
display(exposure_duplicate_distribution)
display(ys0071_audit)
display(unmatched_boundary_out_of_range.head(20))


### S2-01-3. 정제 사건 위치 지도 저장

In [ ]:
cleaned_location_plot = PLOT_DIR / "S2-01_cleaned_fire_location_map.png"

type_palette = {
    "영동 해안형": "#0072B2",
    "영서 내륙형": "#009E73",
    "고지·산간형": "#D55E00",
}

fig, ax = plt.subplots(figsize=(8, 9))
grid.plot(ax=ax, facecolor="#f5f5f5", edgecolor="#8c8c8c", linewidth=0.5)
for climate_name, color in type_palette.items():
    subset = clean_fire_gdf[clean_fire_gdf["기후지형유형"] == climate_name]
    if not subset.empty:
        subset.plot(ax=ax, markersize=11, color=color, alpha=0.72, label=f"{climate_name} (n={len(subset)})")
ax.set_title("S2-01 정제 산불 사건 위치")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.legend(loc="lower left", frameon=True, fontsize=9)
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(cleaned_location_plot, dpi=180, bbox_inches="tight")
plt.show()

print(cleaned_location_plot)


### S2-01-4. 미매칭·경계점 위치 지도 저장

In [ ]:
boundary_location_plot = PLOT_DIR / "S2-01_unmatched_boundary_location_map.png"

raw_unmatched_gdf = gpd.GeoDataFrame(
    raw_spatial_unmatched.copy(),
    geometry=gpd.points_from_xy(raw_spatial_unmatched["경도"], raw_spatial_unmatched["위도"]) if len(raw_spatial_unmatched) else [],
    crs="EPSG:4326",
)
boundary_points_plot = boundary_points_wgs84.copy()

fig, ax = plt.subplots(figsize=(8, 9))
grid.plot(ax=ax, facecolor="#f7f7f7", edgecolor="#9a9a9a", linewidth=0.5)
if not boundary_points_plot.empty:
    boundary_points_plot.plot(
        ax=ax,
        markersize=28,
        color="#E69F00",
        alpha=0.85,
        label=f"경계 {int(BOUNDARY_DISTANCE_M)}m 이내 (n={len(boundary_points_plot)})",
    )
if not raw_unmatched_gdf.empty:
    raw_unmatched_gdf.plot(
        ax=ax,
        markersize=42,
        color="#CC79A7",
        marker="x",
        linewidth=1.5,
        label=f"원본 미매칭 (n={len(raw_unmatched_gdf)})",
    )
if boundary_points_plot.empty and raw_unmatched_gdf.empty:
    ax.text(0.5, 0.5, "미매칭·경계점 0건", transform=ax.transAxes, ha="center", va="center", fontsize=12)
ax.set_title("S2-01 미매칭·경계점 위치")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.legend(loc="lower left", frameon=True, fontsize=9)
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(boundary_location_plot, dpi=180, bbox_inches="tight")
plt.show()

print(boundary_location_plot)


### S2-01-5. 산출물 목록과 최종 감사 통과 확인

In [ ]:
artifact_manifest = pd.DataFrame(
    [
        {
            "artifact_type": "table",
            "filename": filename,
            "path": str(TABLE_DIR / filename),
            "rows": len(df),
            "columns": len(df.columns),
        }
        for filename, df in tables.items()
    ]
    + [
        {
            "artifact_type": "plot",
            "filename": cleaned_location_plot.name,
            "path": str(cleaned_location_plot),
            "rows": np.nan,
            "columns": np.nan,
        },
        {
            "artifact_type": "plot",
            "filename": boundary_location_plot.name,
            "path": str(boundary_location_plot),
            "rows": np.nan,
            "columns": np.nan,
        },
    ]
)
artifact_manifest.to_csv(TABLE_DIR / "S2-01_artifact_manifest.csv", index=False, encoding="utf-8-sig")

fatal_failed_checks = key_quality_audit.loc[~key_quality_audit["pass"]].copy()
display(artifact_manifest)
display(fatal_failed_checks)
assert fatal_failed_checks.empty, fatal_failed_checks.to_string(index=False)

print(f"{ANALYSIS_ID} execution complete")


## S2-01 종료 / S2-02 추가 위치

S2-02 월별·시간대별·권역별 발생 분포 코드는 이 아래에 추가한다.